In [1]:
from pathlib import Path
import json
from lstm_translator import BPETokenizer, iter_parallel_rows, seed_everything, TypoGenerator
from lstm_translator.typo import tokenizer_training_texts

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()


/home/unicorn/my-lstm-translator/.venv-3-12/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Select the intended translation corpus explicitly; do not fall back to fra.txt.
data_file_name = ROOT / "data/csv/en-fr.csv"
OUTPUT_DIR = ROOT / "artifacts/tokenizers/noisy-v1"
SEED = 42
TOKENIZER_WORKERS = 10  # parallel word/pair counting; use 1 for small corpora
TOKENIZER_CHUNK_SIZE = 500  # extraction progress updates after each completed chunk
EN_MERGES = 12_000  # previousliy 6,500; actual vocabulary also includes characters
FR_MERGES = 10_000  # previously 7,000
NOISE_PROBABILITY = 0.23
NOISE_OPTIONS = dict(aug_char_p=0.1, aug_word_p=0.1, aug_char_max=1, aug_word_max=3)
assert data_file_name.is_file(), f"Set data_file_name to your corpus: {data_file_name}"
print("Corpus:", data_file_name.resolve())
print("Output:", OUTPUT_DIR)
print("Merge budgets:", EN_MERGES, FR_MERGES)


Corpus: /home/unicorn/my-lstm-translator/data/csv/en-fr.csv
Output: /home/unicorn/my-lstm-translator/artifacts/tokenizers/noisy-v1
Merge budgets: 12000 10000


In [3]:
seed_everything(SEED)
en_tokenizer = BPETokenizer()
noise = TypoGenerator(corruption_probability=NOISE_PROBABILITY, lang="en", **NOISE_OPTIONS)
# Preserve all clean English and add sampled noisy variants; no materialized corpus.
en_tokenizer.train(tokenizer_training_texts(
    (english for english, _ in iter_parallel_rows(data_file_name)), noise,
), n_merges=EN_MERGES, num_workers=TOKENIZER_WORKERS, chunk_size=TOKENIZER_CHUNK_SIZE)
print("English vocabulary:", len(en_tokenizer.tokens))


Extracting words: 22520376texts [09:30, 39474.06texts/s]
100%|██████████| 12000/12000 [01:14<00:00, 160.04it/s]


English vocabulary: 14927


In [4]:
# French is the clean prediction target; only the English input needs typo variants.
fr_tokenizer = BPETokenizer()
fr_tokenizer.train((french for _, french in iter_parallel_rows(data_file_name)), n_merges=FR_MERGES, num_workers=TOKENIZER_WORKERS, chunk_size=TOKENIZER_CHUNK_SIZE)
print("French vocabulary:", len(fr_tokenizer.tokens))


Extracting words: 22520376texts [03:17, 113921.00texts/s]
100%|██████████| 10000/10000 [00:47<00:00, 212.50it/s]


French vocabulary: 12836


In [7]:
seed_everything(SEED)
preview_noise = TypoGenerator(corruption_probability=1, **NOISE_OPTIONS)
for clean in ["I am watching a beautiful sunset!", "Please translate this sentence."]:
    noisy = preview_noise(clean)
    print({"clean": en_tokenizer.encode(clean), "noisy": en_tokenizer.encode(noisy),
           "clean_tokens": len(en_tokenizer.encode(clean)),
           "noisy_tokens": len(en_tokenizer.encode(noisy))})


{'clean': ['I</w>', ' am</w>', ' w', 'atch', 'ing</w>', ' a</w>', ' be', 'au', 'tif', 'ul</w>', ' sun', 'set</w>', '!</w>'], 'noisy': ['I</w>', ' am</w>', ' wat', 'c', 'Y', 'ing</w>', ' a</w>', ' be', 'au', 'tif', 'ul</w>', ' sun', 'set</w>', '!</w>'], 'clean_tokens': 13, 'noisy_tokens': 14}
{'clean': ['Please</w>', ' transl', 'ate</w>', ' this</w>', ' sentence</w>', '.</w>'], 'noisy': ['Pl', 'eas', 's</w>', ' transl', 'ate</w>', ' this</w>', ' sentence</w>', '.</w>'], 'clean_tokens': 6, 'noisy_tokens': 8}


In [6]:
from lstm_translator.diagnostics import tokenizer_info
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
en_tokenizer.save(OUTPUT_DIR / "en.json")
fr_tokenizer.save(OUTPUT_DIR / "fr.json")
metadata = dict(corpus=str(data_file_name.resolve()), corpus_size=data_file_name.stat().st_size,
                seed=SEED, en_merges=EN_MERGES, fr_merges=FR_MERGES,
                noise_probability=NOISE_PROBABILITY, noise_options=NOISE_OPTIONS,
                source_tokenizer=tokenizer_info(en_tokenizer),
                target_tokenizer=tokenizer_info(fr_tokenizer))
(OUTPUT_DIR / "training.json").write_text(json.dumps(metadata, indent=2) + "\n")
print("Saved:", OUTPUT_DIR)
print("Re-filter the corpus with these tokenizers, then train a NEW model using these paths.")


Saved: /home/unicorn/my-lstm-translator/artifacts/tokenizers/noisy-v1
Re-filter the corpus with these tokenizers, then train a NEW model using these paths.
